# Unix Shell Tutorial: Filtering and Extracting Biomedical Data

This is the **third tutorial** in a series that will demonstrate how shell scripting can be used to perform the tasks that health and life science specialists may need to undertake to find and retrieve biomedical data and text. We will use the compound caffeine as an example and explore different public repositories to identify diseases related to it. The focus is not on the specific relationships we may discover, but on the process of obtaining them.

The objective of this tutorial is to learn how to efficiently filter and extract relevant data from the CSV file retrieved in the previous tutorial. Specifically, we will focus on filtering for proteins associated with putative caffeine-related diseases and extracting only the corresponding protein identifiers.

> This tutorial is part of a series of tutorials adapted as interactive versions of the hands-on steps described in the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book, which is licensed under the [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## Step 1: Filtering Relevant Data with `grep`

Some data in the CSV file may not be relevant regarding our information need, i.e. we may need to identify and extract relevant data. In our case, we will select the relevant proteins (lines) using the command line tool `grep`, and secondly, we will select the column we need using the command line tool `cut`. Since our information need is about diseases related to caffeine, we may assume that we are only interested in proteins from specific species (HUMAN, RAT, and MOUSE) that appear in the protein identifier column.

Extracting lines from a text file is the main function of `grep`. The selection is performed by giving as input a pattern that grep tries to find in each line, presenting only the ones where it was able to find a match. The pattern is the same as the one we normally use when searching for a word in our text editor. The grep command also works with more complex patterns such as regular expressions, that we will describe later on.

To get started, we first need to retrieve the data file generated in the previous tutorial. The following command downloads the `chebi_27732_xrefs_UniProt.csv` file directly from the GitHub repository:

In [1]:
%%bash
curl -s -O 'https://raw.githubusercontent.com/lasigeBioTM/data-text-processing-notebooks/refs/heads/main/data/chebi_27732_xrefs_UniProt.csv'

### Single and Multiple Patterns

We can execute the following command that selects the proteins containing `RAT` as the species identifier in our CSV file:

In [2]:
%%bash
grep 'RAT' chebi_27732_xrefs_UniProt.csv

"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"
"27732","CHEBI:27732","chebi","PUP1_ARATH","Q9FZ96","uniprot"
"27732","CHEBI:27732","chebi","RYR2_RAT","B0LPN4","uniprot"


**Expected Output:** A shorter list of proteins, all containing `RAT` as the species identifier.

The `data` folder contains the files retrieved in the previous tutorial.

## Step 2: Multiple Pattern Matching

To use multiple patterns, we must precede each pattern with the `-e` option:

In [3]:
%%bash
grep -e 'HUMAN' -e 'RAT' -e 'MOUSE' chebi_27732_xrefs_UniProt.csv

"27732","CHEBI:27732","chebi","RYR1_MOUSE","E9PZQ0","uniprot"
"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"
"27732","CHEBI:27732","chebi","RYR1_HUMAN","P21817","uniprot"
"27732","CHEBI:27732","chebi","CP3A4_HUMAN","P08684","uniprot"
"27732","CHEBI:27732","chebi","ATR_HUMAN","Q13535","uniprot"
"27732","CHEBI:27732","chebi","PUP1_ARATH","Q9FZ96","uniprot"
"27732","CHEBI:27732","chebi","SMG1_HUMAN","Q96Q15","uniprot"
"27732","CHEBI:27732","chebi","SMG1_MOUSE","Q8BKX6","uniprot"
"27732","CHEBI:27732","chebi","RYR2_HUMAN","Q92736","uniprot"
"27732","CHEBI:27732","chebi","RYR2_MOUSE","E9Q401","uniprot"
"27732","CHEBI:27732","chebi","PNKD_HUMAN","Q8N490","uniprot"
"27732","CHEBI:27732","chebi","ATR_MOUSE","Q9JKK8","uniprot"
"27732","CHEBI:27732","chebi","CP1A2_HUMAN","P05177","uniprot"
"27732","CHEBI:27732","chebi","RYR3_HUMAN","Q15413","uniprot"
"27732","CHEBI:27732","chebi","RYR2_RAT","B0LPN4","uniprot"
"27732","CHEBI:27732","chebi","RYR3_MOUSE","A2AGL3","uniprot"


**Expected Output:** A longer list of proteins matching any of the three species (HUMAN, RAT, or MOUSE).

The equivalent long form to the `-e` option is `--regexp=PATTERN`.

In a terminal, we would typically use `| less` to scroll through long outputs. However, in a notebook environment, using `less` does not work well because it requires interactive input. Instead, we can use the `head` command to preview just the first few lines of the output. The `-n 10` option tells `head` to show only the first 10 lines.

In [4]:
%%bash
grep -e 'HUMAN' -e 'RAT' -e 'MOUSE' chebi_27732_xrefs_UniProt.csv | head -n 10

"27732","CHEBI:27732","chebi","RYR1_MOUSE","E9PZQ0","uniprot"
"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"
"27732","CHEBI:27732","chebi","RYR1_HUMAN","P21817","uniprot"
"27732","CHEBI:27732","chebi","CP3A4_HUMAN","P08684","uniprot"
"27732","CHEBI:27732","chebi","ATR_HUMAN","Q13535","uniprot"
"27732","CHEBI:27732","chebi","PUP1_ARATH","Q9FZ96","uniprot"
"27732","CHEBI:27732","chebi","SMG1_HUMAN","Q96Q15","uniprot"
"27732","CHEBI:27732","chebi","SMG1_MOUSE","Q8BKX6","uniprot"
"27732","CHEBI:27732","chebi","RYR2_HUMAN","Q92736","uniprot"
"27732","CHEBI:27732","chebi","RYR2_MOUSE","E9Q401","uniprot"


**Expected Output:** The first 10 lines of the filtered protein list.

## Creating and Updating the Script

We can now update our script file to contain the following lines:

```bash
url="curl "https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=100&format=csv"

curl -s "$url" | \
    grep \
      -e 'HUMAN' \
      -e 'RAT' \
      -e 'MOUSE'
```

We should note that we added the `-s` option to suppress the progress information of curl, and the characters `| \` to the end of line to redirect the output of that line as input of the next line, in this case the grep command. We need to be careful in ensuring that `\` is the last character in the line, i.e. spaces in the end of the line may cause problems.

> **Note:** The `curl` command here at this platform will be replaced by a local version to access the data locally instead of online due to access restrictions. However, the same command will work just fine in your local terminal.

In [5]:
%%bash
cat > getproteins.sh << 'EOF'
url="https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=100&format=csv"

curl -s "$url" | \
    grep \
      -e 'HUMAN' \
      -e 'RAT' \
      -e 'MOUSE'
EOF

**Expected Output:** Script file `getproteins.sh` created successfully.

In [6]:
%%bash
chmod u+x getproteins.sh

**Expected Output:** No output (permissions set successfully).

In [7]:
%%bash
./getproteins.sh 27732

"27732","CHEBI:27732","chebi","RYR1_MOUSE","E9PZQ0","uniprot"
"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"
"27732","CHEBI:27732","chebi","RYR1_HUMAN","P21817","uniprot"
"27732","CHEBI:27732","chebi","CP3A4_HUMAN","P08684","uniprot"
"27732","CHEBI:27732","chebi","ATR_HUMAN","Q13535","uniprot"
"27732","CHEBI:27732","chebi","PUP1_ARATH","Q9FZ96","uniprot"
"27732","CHEBI:27732","chebi","SMG1_HUMAN","Q96Q15","uniprot"
"27732","CHEBI:27732","chebi","SMG1_MOUSE","Q8BKX6","uniprot"
"27732","CHEBI:27732","chebi","RYR2_HUMAN","Q92736","uniprot"
"27732","CHEBI:27732","chebi","RYR2_MOUSE","E9Q401","uniprot"
"27732","CHEBI:27732","chebi","PNKD_HUMAN","Q8N490","uniprot"
"27732","CHEBI:27732","chebi","ATR_MOUSE","Q9JKK8","uniprot"
"27732","CHEBI:27732","chebi","CP1A2_HUMAN","P05177","uniprot"
"27732","CHEBI:27732","chebi","RYR3_HUMAN","Q15413","uniprot"
"27732","CHEBI:27732","chebi","RYR2_RAT","B0LPN4","uniprot"
"27732","CHEBI:27732","chebi","RYR3_MOUSE","A2AGL3","uniprot"


**Expected Output:** Filtered list of relevant proteins for caffeine (CHEBI:27732).

In [8]:
%%bash
./getproteins.sh 27732 > chebi_27732_xrefs_UniProt_relevant.csv

**Expected Output:** No output (file saved successfully).

## Step 3: Data Elements Selection with `cut`

Now we need to select just the first column, the one that contains the protein identifiers. Selecting columns from a tabular file is one easy task for `cut`. The cut command can receive as arguments the character that divides each data element (column) in a line using the `-d` option, and the `-f` option to indicate which columns to select. The equivalent long form to the `-d` option is `--delimiter=DELIM`. The equivalent long form to the `-f` option is `--fields=LIST`. For example, we can get the first column of our CSV file:

In [9]:
%%bash
cut -d, -f1 < chebi_27732_xrefs_UniProt_relevant.csv

"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"


**Expected Output:** Only the first column of the file (protein identifiers).

We should note that comma (`,`) is the character that separates data elements in a CSV file, and represents the first data element.

The command will display only the first column of the file, i.e. the protein identifiers.

We can also extract multiple columns at once by separating the column numbers with a comma. For example, to get both the first column (the ChEBI identifier) and the fifth column (the UniProt identifier), we use `-f1,5`:

In [10]:
%%bash
cut -d, -f1,5 < chebi_27732_xrefs_UniProt_relevant.csv

"27732","E9PZQ0"
"27732","F1LMY4"
"27732","P21817"
"27732","P08684"
"27732","Q13535"
"27732","Q9FZ96"
"27732","Q96Q15"
"27732","Q8BKX6"
"27732","Q92736"
"27732","E9Q401"
"27732","Q8N490"
"27732","Q9JKK8"
"27732","P05177"
"27732","Q15413"
"27732","B0LPN4"
"27732","A2AGL3"


**Expected Output:** First and fifth columns of the file.

Now, the output contains both the first and fifth column of the file.

## Final Script with Column Selection

We can update our script file to contain the following lines:

```bash
url="https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=100&format=csv"

curl -s "$url" | \
 grep \
 -e 'HUMAN' \
 -e 'RAT' \
 -e 'MOUSE' | \
 cut -d, -f5 | \
 tr -d \"
```

The last two lines have changed from the previous version: we now extract field 5 (the UniProt identifier column) using `cut -d, -f5`, and then use `tr -d \"` to remove the quotation marks from the output, giving us clean protein identifiers.

In [11]:
%%bash
cat > getproteins.sh << 'EOF'
url="https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=100&format=csv"

curl -s "$url" | \
    grep \
      -e 'HUMAN' \
      -e 'RAT' \
      -e 'MOUSE' | \
    cut -d, -f5 | \
    tr -d \"
EOF

**Expected Output:** Final script version created.

Now we can execute the script for caffeine:

In [12]:
%%bash
./getproteins.sh 27732

E9PZQ0
F1LMY4
P21817
P08684
Q13535
Q9FZ96
Q96Q15
Q8BKX6
Q92736
E9Q401
Q8N490
Q9JKK8
P05177
Q15413
B0LPN4
A2AGL3


**Expected Output:** Only protein identifiers (fifth column, with quotes removed) for proteins from HUMAN, RAT, and MOUSE species.

In [13]:
%%bash
./getproteins.sh 27732 > chebi_27732_xrefs_UniProt_relevant_identifiers.csv

**Expected Output:** No output (file saved successfully).

To check if the file was really created and to analyze its contents, we can
use the `cat` command:

In [14]:
%%bash
cat chebi_27732_xrefs_UniProt_relevant_identifiers.csv

E9PZQ0
F1LMY4
P21817
P08684
Q13535
Q9FZ96
Q96Q15
Q8BKX6
Q92736
E9Q401
Q8N490
Q9JKK8
P05177
Q15413
B0LPN4
A2AGL3


**Expected Output:** List of protein identifiers associated with caffeine-related diseases.

# Conclusion

This concludes the **Unix Shell** tutorial adapted from the same section of the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book.

In this tutorial, we learned how to filter data from biomedical databases using cURL and web services.

In the next tutorial of this series will explore task repetition techniques to efficiently apply the same task to all proteins in the list we have gathered.

# Exercise

As an exercise execute the script to extract only the protein identifiers associated with [water](https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:15377) and [gold](https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:30050).

In [15]:
%%bash
./getproteins.sh 15377 > water_proteins.csv

**Expected Output:** File `water_proteins.csv` created with protein identifiers for water (CHEBI:15377).

In [16]:
%%bash
./getproteins.sh 30050 > gold_proteins.csv

**Expected Output:** File `gold_proteins.csv` created with protein identifiers for gold (CHEBI:30050).

In [17]:
%%bash
cat water_proteins.csv

Q84WM7
P36269
Q8VYI3
O55071
P43299
Q9SGU9
Q9EPW0
Q8GS60


**Expected Output:** List of protein identifiers associated with water.

In [18]:
%%bash
cat gold_proteins.csv

Q9M0R2
Q9LIL4
Q96DZ5
Q63524
Q78IS1
Q9Y3A6
Q8WW62
Q8VDC1
Q9BQS8
Q15363
Q8R1V4
Q9S7M9
Q8RWM6
Q8GYG1
Q9SCU1
Q99KF1
Q5I0E7
Q5BK85
Q9R0Q3
Q9Y3Q3
Q6AY25
Q7Z7H5
Q9CXE7
Q6AXN3
Q9CQG0
Q86XR7
Q3UHI4
Q9LJV9
P49755
Q9Y3B3
D3ZTX0
Q56Z59
Q6IDL4
F4J4Y0
O81045
Q56WK6
Q6PL24
Q9BVK6
Q9FVU0
Q9LQY3
Q8VY92
Q92503
Q94C59
Q7TNY6


**Expected Output:** List of protein identifiers associated with gold.